In [14]:
import os
PROJECT = "/content/assignment_pts_7_8_9_full"
os.makedirs(PROJECT, exist_ok=True)
%cd {PROJECT}


/content/assignment_pts_7_8_9_full


In [15]:
from google.colab import files
uploaded = files.upload()  # select the 4 patched notebooks
print("Uploaded:", list(uploaded.keys()))
!ls -la


Saving Task_3_patched.ipynb to Task_3_patched (1).ipynb
Saving Task_2_patched.ipynb to Task_2_patched (1).ipynb
Saving DMML_4_5_6_patched.ipynb to DMML_4_5_6_patched (1).ipynb
Saving Script-7to9_patched.ipynb to Script-7to9_patched (1).ipynb
Uploaded: ['Task_3_patched (1).ipynb', 'Task_2_patched (1).ipynb', 'DMML_4_5_6_patched (1).ipynb', 'Script-7to9_patched (1).ipynb']
total 116
drwxr-xr-x 6 root root  4096 Aug 23 11:41  .
drwxr-xr-x 1 root root  4096 Aug 23 10:53  ..
drwxr-xr-x 5 root root  4096 Aug 23 10:56  data
-rw-r--r-- 1 root root  3294 Aug 23 11:41 'DMML_4_5_6_patched (1).ipynb'
-rw-r--r-- 1 root root  3294 Aug 23 10:54  DMML_4_5_6_patched.ipynb
-rw-r--r-- 1 root root    88 Aug 23 10:57  Feature_Metadata.md
-rw-r--r-- 1 root root 17679 Aug 23 10:58 'full_pipeline_points_1_to_10_submission_20250823_105749 (1).zip'
-rw-r--r-- 1 root root 17679 Aug 23 10:57  full_pipeline_points_1_to_10_submission_20250823_105749.zip
drwxr-xr-x 2 root root  4096 Aug 23 10:56  models
drwxr-xr-x 2

In [16]:
!pip -q install prefect papermill pandas numpy scikit-learn joblib


In [17]:
from textwrap import dedent

flow_code = dedent("""
from prefect import flow, task, get_run_logger
import subprocess, os, json, glob, datetime as dt, pathlib

ART_DIR = "orchestration_artifacts"
pathlib.Path(ART_DIR).mkdir(exist_ok=True)

NOTEBOOKS_IN_ORDER = [
    "Task_2_patched.ipynb",
    "Task_3_patched.ipynb",
    "DMML_4_5_6_patched.ipynb",
    "Script-7to9_patched.ipynb",
]

def run_cmd(cmd):
    p = subprocess.run(cmd, text=True, capture_output=True)
    return p.returncode, p.stdout, p.stderr

@task
def run_notebook(nb_path: str):
    logger = get_run_logger()
    out_nb = os.path.join(ART_DIR, nb_path.replace(".ipynb", "_executed.ipynb"))
    logger.info(f" Starting {nb_path}")
    code, out, err = run_cmd(["papermill", "-k", "python3", nb_path, out_nb])
    if out: logger.info(out)
    if code != 0:
        if err: logger.error(err)
        raise RuntimeError(f"Failed executing notebook: {nb_path}")
    logger.info(f" Finished {nb_path} -> {out_nb}")
    return out_nb

@task
def write_run_summary():
    summary = {
        "run_id": dt.datetime.now().isoformat(),
        "executed_notebooks": sorted(glob.glob(os.path.join(ART_DIR, "*_executed.ipynb"))),
        "raw": sorted(glob.glob("data/raw/**/*", recursive=True))[:20],
        "validated": sorted(glob.glob("data/validated/*")),
        "transformed": sorted(glob.glob("data/transformed/*")),
        "feature_store": sorted(glob.glob("feature_store/feature_data/*"))[-10:],
        "models": sorted(glob.glob("models/*.pkl")),
        "reports": sorted(glob.glob("reports/*.json")),
    }
    out = os.path.join(ART_DIR, "orchestration_run_summary.json")
    with open(out, "w") as f: json.dump(summary, f, indent=2)
    return out

@flow(name="pipeline-1-to-9-orchestration")
def run_pipeline():
    for nb in NOTEBOOKS_IN_ORDER:
        run_notebook(nb)
    summary_path = write_run_summary()
    get_run_logger().info(f"Run summary written to: {summary_path}")

if __name__ == "__main__":
    run_pipeline()
""")

with open("prefect_flow.py", "w") as f:
    f.write(flow_code)

print(" Wrote prefect_flow.py")


 Wrote prefect_flow.py


In [18]:
!python prefect_flow.py


11:41:57.075 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8071
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.
11:42:03.918 | INFO    | Flow run 'deft-skua' - Beginning flow run 'deft-skua' for flow 'pipeline-1-to-9-orchestration'
11:42:04.056 | INFO    | Task run 'run_notebook-63b' -  Starting Task_2_patched.ipynb
11:42:08.682 | INFO    | Task run 'run_notebook-63b' -  Finished Task_2_patched.ipynb -> orchestration_artifacts/Task_2_patched_executed.ipynb
11:42:08.686 | INFO    | Task run 'run_notebook-63b' - Finished in state Completed()
11:42:08.768 | INFO    | Task run 'run_notebook-06a' -  Starting Task_3_patched.ipynb
11:42:14.478 | INFO    | Task run 'run_notebook-06a' -  Finished Task_3_patched.ipynb -> orchestration_artifacts/Task_3_patched_executed.ipynb
11:42:14.483 | INFO    | Task run 'run_notebook-06a' - Finished in state Completed()
11:42:14.610 | INFO    | Task run 'run_notebo

In [19]:
!ls -la models
!ls -la reports
!sed -n '1,200p' orchestration_artifacts/orchestration_run_summary.json


total 140
drwxr-xr-x 2 root root   4096 Aug 23 10:56 .
drwxr-xr-x 6 root root   4096 Aug 23 11:41 ..
-rw-r--r-- 1 root root   1948 Aug 23 11:42 logreg.pkl
-rw-r--r-- 1 root root 128685 Aug 23 11:42 random_forest.pkl
total 24
drwxr-xr-x 2 root root 4096 Aug 23 10:56 .
drwxr-xr-x 6 root root 4096 Aug 23 11:41 ..
-rw-r--r-- 1 root root  139 Aug 23 11:42 evaluation_logreg.json
-rw-r--r-- 1 root root  146 Aug 23 11:42 evaluation_random_forest.json
-rw-r--r-- 1 root root  342 Aug 23 11:42 retrieved_features_sample.csv
-rw-r--r-- 1 root root   19 Aug 23 11:42 validation_report.txt
{
  "run_id": "2025-08-23T11:42:28.247000",
  "executed_notebooks": [
    "orchestration_artifacts/DMML_4_5_6_patched_executed.ipynb",
    "orchestration_artifacts/Script-7to9_patched_executed.ipynb",
    "orchestration_artifacts/Task_2_patched_executed.ipynb",
    "orchestration_artifacts/Task_3_patched_executed.ipynb"
  ],
  "raw": [
    "data/raw/generic",
    "data/raw/generic/generic_churn_sample.csv",
    "dat

In [21]:
# Write Problem_Formulation.md
problem_md = """# Problem Formulation — Customer Churn Prediction Pipeline
... (use text I shared earlier) ...
"""
with open("Problem_Formulation.md", "w") as f:
    f.write(problem_md)

# Write Feature_Metadata.md
feature_md = """# Feature Metadata — Customer Churn Feature Store
... (use text I shared earlier) ...
"""
with open("Feature_Metadata.md", "w") as f:
    f.write(feature_md)


In [22]:
#  Run this inside: /content/assignment_pts_7_8_9_full
import os, glob, zipfile
from datetime import datetime

PROJECT = "/content/assignment_pts_7_8_9_full"
os.chdir(PROJECT)

# Collect key artifacts from Points 1–10
files_explicit = [
    "prefect_flow.py",
    "Problem_Formulation.md",
    "Feature_Metadata.md",
    "orchestration_artifacts/orchestration_run_summary.json",
    "orchestration_artifacts/Task_2_patched_executed.ipynb",
    "orchestration_artifacts/Task_3_patched_executed.ipynb",
    "orchestration_artifacts/DMML_4_5_6_patched_executed.ipynb",
    "orchestration_artifacts/Script-7to9_patched_executed.ipynb",
    "reports/retrieved_features_sample.csv",
    "reports/validation_report.txt",
]
files_dynamic = sorted(glob.glob("reports/evaluation_*.json")) + sorted(glob.glob("models/*.pkl"))
to_zip = [p for p in (files_explicit + files_dynamic) if os.path.exists(p)]

# Build README safely as a normal string
readme = "# Full Pipeline Submission — Points 1 to 10\n\n"
readme += "This archive contains all key artifacts and evidence from the complete pipeline.\n\n"
readme += "## How to Run\n```\n"
readme += "pip install prefect papermill pandas numpy scikit-learn joblib\n"
readme += "python prefect_flow.py\n```\n\n"
readme += "## Key Artifacts\n"
readme += "- Orchestrator: `prefect_flow.py`\n"
readme += "- Run evidence JSON: `orchestration_artifacts/orchestration_run_summary.json`\n"
readme += "- Executed notebooks: `orchestration_artifacts/*_executed.ipynb`\n"
readme += "- Point 7 proof: `reports/retrieved_features_sample.csv`\n"
readme += "- Validation report: `reports/validation_report.txt`\n"
readme += "- Models (Point 9): `models/*.pkl`\n"
readme += "- Evaluation metrics: `reports/evaluation_*.json`\n\n"
readme += "## Notes\n"
readme += "- Data files under `data/` are excluded to keep the archive small.\n"
readme += f"- Created on: {datetime.now().isoformat()}\n"

# Make zip
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
zipname = f"full_pipeline_points_1_to_10_submission_{ts}.zip"
with zipfile.ZipFile(zipname, "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr("README.md", readme)
    for p in to_zip:
        z.write(p)
        print("Added:", p)

print("\n Created:", zipname, "| Files added:", len(to_zip), "+ README.md")

from google.colab import files
files.download(zipname)


Added: prefect_flow.py
Added: Problem_Formulation.md
Added: Feature_Metadata.md
Added: orchestration_artifacts/orchestration_run_summary.json
Added: orchestration_artifacts/Task_2_patched_executed.ipynb
Added: orchestration_artifacts/Task_3_patched_executed.ipynb
Added: orchestration_artifacts/DMML_4_5_6_patched_executed.ipynb
Added: orchestration_artifacts/Script-7to9_patched_executed.ipynb
Added: reports/retrieved_features_sample.csv
Added: reports/validation_report.txt
Added: reports/evaluation_logreg.json
Added: reports/evaluation_random_forest.json
Added: models/logreg.pkl
Added: models/random_forest.pkl

 Created: full_pipeline_points_1_to_10_submission_20250823_114255.zip | Files added: 14 + README.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
from google.colab import files
uploaded = files.upload()  # Select your full_pipeline_points_1_to_10_submission_20250823_092548.zip


Saving full_pipeline_points_1_to_10_submission_20250823_114255.zip to full_pipeline_points_1_to_10_submission_20250823_114255 (1).zip


In [24]:
import zipfile

zip_path = "full_pipeline_points_1_to_10_submission_20250823_114255.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    print(" ZIP Contents:\n")
    for f in z.namelist():
        print(f)


 ZIP Contents:

README.md
prefect_flow.py
Problem_Formulation.md
Feature_Metadata.md
orchestration_artifacts/orchestration_run_summary.json
orchestration_artifacts/Task_2_patched_executed.ipynb
orchestration_artifacts/Task_3_patched_executed.ipynb
orchestration_artifacts/DMML_4_5_6_patched_executed.ipynb
orchestration_artifacts/Script-7to9_patched_executed.ipynb
reports/retrieved_features_sample.csv
reports/validation_report.txt
reports/evaluation_logreg.json
reports/evaluation_random_forest.json
models/logreg.pkl
models/random_forest.pkl
